## Init
### Imports

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from pypdf import PdfReader
from unidecode import unidecode
from langchain.text_splitter import RecursiveCharacterTextSplitter
import faiss
import numpy as np
import pickle
# from langchain_community.document_loaders import PyPDFLoader
import os

# Local imports
from classes.Document import Document

### Configuration

In [2]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(
    api_key=openai_api_key
)

### Test

In [3]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="I like cats and dogs",
)
print(len(response.data[0].embedding))
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="I like cats and dogs and I like to play with them every day.\nI also enjoy taking them for walks in the park and giving them treats.\nThey are my favorite pets and I love spending time with them.",
)
np_embedding = np.array(response.data[0].embedding, dtype=np.float32)
print(np_embedding.shape)

1536
(1536,)


El len de los embeddings es de 1536

## Dataset Preprocessing

In [4]:
authors = []
with open("authors.txt", "r") as f:
    for line in f:
        author = line.strip().split(' ')
        # author = author[2:] + author[:2]
        # author = ' '.join(author)
        authors.append(author)
        print(author)

['Almendares', 'Barrantes', 'Dillan']
['Barquero', 'Diaz', 'Pablo']
['Brenes', 'Rodriguez', 'Rayforth', 'Josue']
['Calderon', 'Cubero', 'Brandon', 'Adones']
['Calderon', 'Diaz', 'Ignacio']
['Chacon', 'Beckford', 'Jeremy', 'De', 'Jesus']
['Chacon', 'Rivas', 'Kenneth', 'Fabian']
['Fernandez', 'Martinez', 'Mariana']
['Fernandez', 'Murillo', 'Hans']
['Garbanzo', 'Carvajal', 'Daniel', 'Alonso']
['Gonzalez', 'Lopez', 'Pamela', 'Maria']
['Granados', 'Preciado', 'Tomas']
['Granados', 'Siles', 'Pablo']
['Herrera', 'Campos', 'Israel', 'Antonio']
['Martinez', 'Vargas', 'Andres', 'Felipe']
['Montero', 'Rodriguez', 'Joselyn', 'Vanessa']
['Navarro', 'Todd', 'Luis', 'Carlos']
['Perez', 'Machado', 'Steven', 'David']
['Perez', 'Picado', 'Esteban']
['Rivera', 'Serrano', 'Marco', 'Vinicio']
['Rodriguez', 'Murillo', 'Manuel', 'Alejandro']
['Sanabria', 'Calvo', 'Diana', 'Valeria']
['Sanabria', 'Marroquin', 'Raul']
['Sanchez', 'Cespedes', 'John', 'Stuart']
['Sandi', 'Barrantes', 'Victoria']
['Sandoval', 'Sa

In [5]:
class File:
    def __init__(self, filename):
        # Initial metadata
        self.filename = filename
        filename_parts = filename[:-4].split('_')
        self.week = filename_parts[0]
        date = filename_parts[3]
        self.anomaly = False
        self.__set_date(date)
        self.author = self.__extract_author()
        if author == "Unknown Author":
            print(f"Author not found for file: {self.filename}")
        # print(f"File created: {self.filename}, Week: {self.week}, Date: {self.date}, Author: {self.author}")

    def __set_date(self, date):
        self.date = f"{date[:2]}-{date[2:4]}-{date[4:]}"

    def __extract_author(self):
        # Extract author from the PDF
        file = PdfReader(f"dataset/{self.filename}")
        page = file.pages[0]
        content = page.extract_text(extraction_mode='plain').split('\n')
        if len(content) > 100:
            self.anomaly = True
        count = 0
        for index, line in enumerate(content):
            line = line.replace(' ´', '')
            line = unidecode(line.strip())
            if index > 30:
                return "Unknown Author"
            for author in authors:
                for word in author:
                    if word in line:
                        count += 1
                if count >= 2:
                    author = author[2:] + author[:2]
                    author = ' '.join(author)
                    return author
                
    def get_chunks(self, chunk_size=1000, chunk_overlap=200):

        # Use PdfReader to load the PDF and extract text
        file = PdfReader(f"dataset/{self.filename}")
        # Combine all page texts into a single string
        full_text = f"Documento: {self.filename}, hecho por: {self.author}, de la clase del: {self.date}, semana #{self.week} del periodo lectivo. "
        full_text = f""
        for page in file.pages:
            text = page.extract_text()
            text = text.strip()
            text = text.replace(' ´', '')
            text = text.replace(' \'', '')
            text = text.replace(' `', '')
            if text:
                # Normalize text to remove accents and special characters
                full_text += text + " "
        
        # Split text into chunks suitable for RAG/vector DB
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )
        chunks = splitter.split_text(full_text)
        return chunks

In [6]:
file_list = [File(file) for file in os.listdir("dataset")]
file_list.sort(key=lambda x: x.date)
documents = []
embeddings = []
for file in file_list:
    file_chunks = file.get_chunks()
    for chunk in file_chunks:
        new_doc = Document(chunk, file.filename, file.author, file.date, file.week)
        documents.append(new_doc)
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=chunk,
        )
        embedding = np.array(response.data[0].embedding, dtype=np.float32)
        embeddings.append(embedding)

print(f"Number of files in dataset: {len(file_list)}")
print(f"Number of chunks created: {len(documents)}")
np_embeddings = np.array(embeddings, dtype=np.float32)


Number of files in dataset: 52
Number of chunks created: 640


### Create FlatL2 Faiss Index and Save It

In [7]:
# Create a FAISS index
dimension = embeddings[0].shape[0]  # Dimension of the embeddings
index = faiss.IndexFlatL2(dimension)  # L2 distance index
index.add(np_embeddings)  # Add the embeddings to the index
# Save the index to a file
faiss.write_index(index, "faiss_index.index")
# Save the documents to a pickle file
with open("documents.pkl", "wb") as f:
    pickle.dump(documents, f)

### Create Inner Product Faiss Index and Save It

In [8]:
index2 = faiss.IndexFlatIP(dimension)  # Inner product index
index2.add(np_embeddings)  # Add the embeddings to the index
# Save the inner product index to a file
faiss.write_index(index2, "faiss_index_inner_product.index")